# Bonus 3 · EDA Cruzado — Toda la Capa Gold

Análisis exploratorio integrado sobre las 7 tablas Gold del proyecto Pulso Medellín.

| Tabla Gold | Pregunta |
|------------|----------|
| `afluencia_vs_pm25` | ¿La contaminación y la lluvia afectan el uso del Metro? |
| `accidentalidad_por_comuna` | ¿Qué comunas concentran más y más graves accidentes? |
| `demanda_encicla_vs_clima` | ¿El clima reduce el uso de EnCicla? |
| `corredores_riesgo_compuesto` | ¿Qué corredores tienen alto volumen Y alta siniestralidad? |
| `percentiles_metro` | ¿Cuál es la afluencia típica por franja horaria y línea? |
| `red_metro_pagerank` | ¿Qué estaciones son más centrales en la red Metro? |
| `ml_fatalidad_evaluacion` | Métricas del modelo predictivo de gravedad |

> **Requisito previo:** `make pipeline-batch && make pipeline-sprint5`

In [ ]:
import sys
sys.path.insert(0, "/workspace/src")

from shared.config import (
    crear_spark_session,
    TBL_GOLD_AFLUENCIA_PM25,
    TBL_GOLD_ACCIDENTALIDAD,
    TBL_GOLD_ENCICLA_CLIMA,
    TBL_GOLD_CORREDORES_RIESGO,
    TBL_GOLD_PERCENTILES_METRO,
    TBL_GOLD_RED_METRO_PAGERANK,
    TBL_GOLD_ML_FATALIDAD_EVAL,
)
from pyspark.sql import functions as F

spark = crear_spark_session("Notebook-EDA-Completo")
spark.sparkContext.setLogLevel("WARN")
print("SparkSession lista.")

## 1. Inventario de tablas Gold

In [ ]:
tablas = spark.sql("SHOW TABLES IN demo.pulsomed.gold")
tablas.show(truncate=False)

## 2. B-1 · Afluencia Metro vs PM2.5 y precipitación

In [ ]:
af_pm25 = spark.table(TBL_GOLD_AFLUENCIA_PM25)
print(f"Registros en afluencia_vs_pm25: {af_pm25.count():,}")
af_pm25.orderBy("anio", "mes").show(12)

In [ ]:
# Correlación promedio PM2.5 ↔ pasajeros por línea
print("Correlación media PM2.5 ↔ pasajeros por línea:")
(
    af_pm25.groupBy("linea")
    .agg(
        F.avg("corr_pm25_pasajeros").alias("corr_pm25_media"),
        F.avg("corr_precip_pasajeros").alias("corr_precip_media"),
        F.count("*").alias("meses_observados"),
    )
    .orderBy("linea")
    .show()
)

## 3. B-2 · Accidentalidad por comuna

In [ ]:
acc = spark.table(TBL_GOLD_ACCIDENTALIDAD)
print(f"Registros: {acc.count():,}")

# Top 10 comunas por índice de severidad (promedio histórico)
print("\nTop 10 comunas más peligrosas (índice de severidad promedio):")
(
    acc.groupBy("comuna")
    .agg(
        F.sum("incidentes_total").alias("incidentes_totales"),
        F.sum("con_muertos").alias("total_muertos"),
        F.sum("con_heridos").alias("total_heridos"),
        F.avg("indice_severidad").alias("severidad_promedio"),
    )
    .orderBy(F.desc("total_muertos"))
    .show(10, truncate=False)
)

In [ ]:
# Evolución anual de muertos viales en toda la ciudad
print("Evolución anual de muertos en incidentes viales:")
(
    acc.groupBy("anio")
    .agg(F.sum("con_muertos").alias("muertos_total"))
    .orderBy("anio")
    .show()
)

## 4. B-3 · Demanda EnCicla vs clima

In [ ]:
enc = spark.table(TBL_GOLD_ENCICLA_CLIMA)
print(f"Registros: {enc.count():,}")

# Impacto de la lluvia en viajes EnCicla
print("\nViajes EnCicla con lluvia vs sin lluvia:")
(
    enc.groupBy("llovio")
    .agg(
        F.avg("viajes").alias("viajes_promedio"),
        F.avg("viajes_relativos_pct").alias("pct_respecto_media"),
        F.count("*").alias("dias"),
    )
    .orderBy("llovio")
    .show()
)

In [ ]:
# Demanda por bin de temperatura
print("Demanda EnCicla por rango de temperatura:")
(
    enc.groupBy("bin_temperatura")
    .agg(F.avg("viajes").alias("viajes_promedio"))
    .orderBy("bin_temperatura")
    .show()
)

## 5. B-4 · Corredores de riesgo compuesto

In [ ]:
riesgo = spark.table(TBL_GOLD_CORREDORES_RIESGO)
print("Top 10 corredores de mayor riesgo compuesto (volumen × severidad):")
riesgo.orderBy("score_riesgo").show(10, truncate=False)

## 6. Percentiles Metro por franja horaria

In [ ]:
perc = spark.table(TBL_GOLD_PERCENTILES_METRO)
print(f"Registros: {perc.count()}")
print("\nPercentiles de afluencia Metro (línea × franja):")
perc.orderBy("linea", "franja_horaria").show(20, truncate=False)

## 7. Red Metro — PageRank de estaciones

In [ ]:
pr = spark.table(TBL_GOLD_RED_METRO_PAGERANK)
print("Ranking de centralidad PageRank — estaciones Metro:")
pr.orderBy("ranking").select(
    "ranking", "nombre", "linea", "pagerank"
).show(15, truncate=False)

In [ ]:
# Las estaciones de intercambio deberían tener mayor PageRank
print("\nEstaciones de intercambio (línea contiene '/'):")
from pyspark.sql.functions import instr
pr.filter(F.col("linea").contains("/")).orderBy("ranking").show(truncate=False)

## 8. Modelo ML — métricas de evaluación

In [ ]:
ml_eval = spark.table(TBL_GOLD_ML_FATALIDAD_EVAL)
print("Métricas del RandomForestClassifier (Módulo 06a):")
ml_eval.select(
    "modelo", "n_arboles", "max_profundidad",
    "accuracy", "f1_weighted", "precision_weighted", "recall_weighted"
).show(truncate=False)

## 9. Análisis cruzado: accidentes fatales vs calidad del aire

In [ ]:
# ¿Los meses con peor calidad del aire tienen más muertos viales?
# Cruzamos afluencia_vs_pm25 (PM2.5 mensual) con accidentalidad_por_comuna (muertos por año-mes aprox)

pm25_anio = (
    af_pm25.groupBy("anio")
    .agg(F.avg("pm25_promedio_mes").alias("pm25_promedio_anual"))
)

muertos_anio = (
    acc.groupBy("anio")
    .agg(F.sum("con_muertos").alias("muertos_total"))
)

cruce = pm25_anio.join(muertos_anio, on="anio", how="inner").orderBy("anio")
print("Correlación anual PM2.5 ↔ muertos viales:")
cruce.show()

# Correlación de Pearson
corr_val = cruce.stat.corr("pm25_promedio_anual", "muertos_total")
print(f"\nCorrelación de Pearson (PM2.5 anual ↔ muertos viales): {corr_val:.4f}")

## Conclusiones del EDA cruzado

1. **Calidad del aire ↔ afluencia Metro:** las correlaciones por línea muestran si los
   ciudadanos prefieren el Metro cuando la calidad del aire es peor (hipótesis: 
   evitan circular en auto).

2. **Accidentalidad:** las comunas del norte y noroccidente históricamente concentran
   más incidentes fatales; la tendencia anual permite evaluar la efectividad de
   políticas de seguridad vial.

3. **EnCicla vs clima:** la lluvia reduce significativamente los viajes; los días
   con PM2.5 alto también muestran caídas de demanda (efecto combinado salud + clima).

4. **Red Metro — PageRank:** las estaciones de intercambio (San Antonio, Acevedo,
   San Javier) obtienen el mayor PageRank — son los nodos más críticos para la
   resiliencia de la red ante interrupciones.

5. **ML vs EDA:** el modelo RandomForest confirma el EDA: la clase de accidente
   y la hora son las variables más predictivas de la gravedad.